# Paired wavelet experiment: both ears, all subjects, original trial folds
Compare baseline and a predeclared experimental wavelet artifact attenuator, with both retain/exclude training policies. Primary scores use identical nonsuspected validation trials. Full validation and suspicious-trial diagnostics are also saved. No settings are selected by these scores.

This is an exploratory channel-wise method appropriate for two-channel inputs, not a validated clean-EEG reconstruction. Symlet-4 stationary wavelets avoid decimation; seven levels at 500 Hz cover details down to approximately 1.95 Hz. Reflect padding (1024 samples each side, then to a multiple of 128) limits wraparound. Per-trial/channel/scale median +/- 6 robust SD caps only extreme detail coefficients; the coarsest approximation is preserved. This is **upper-tail artifact attenuation**, not conventional suppression of small white-noise coefficients. Sustained contamination and activity below 1.95 Hz may remain. The 6-MAD rule is a fixed conservative experimental choice, not a literature-validated optimum.

Process each full ten-second trial separately before the existing feature bank and SingLEM preparation. No cross-trial threshold fitting, rereferencing, amplitude normalization, extra notch, or automatic trial deletion. Unknown physical units remain an assumption. Scaler, selection, balancing and classifier fitting stay inside training folds.

Method rationale: [single-channel wavelet comparison](https://pmc.ncbi.nlm.nih.gov/articles/PMC4991686/), [low-electrode HAPPILEE](https://pmc.ncbi.nlm.nih.gov/articles/PMC9395507/), [PyWavelets SWT](https://pywavelets.readthedocs.io/en/latest/ref/swt-stationary-wavelet-transform.html). This implementation is an adaptation, not a reproduction of those pipelines.


In [ ]:
import time as _stage_time
print('START cell 2',flush=True)
_stage_start=_stage_time.perf_counter()
import importlib.util, subprocess, sys
packages={'pywt':'PyWavelets','numpy':'numpy','scipy':'scipy','pandas':'pandas','sklearn':'scikit-learn','matplotlib':'matplotlib','xgboost':'xgboost'}
missing=[p for m,p in packages.items() if importlib.util.find_spec(m) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
import io,json,hashlib,platform,warnings,zipfile
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.signal import butter,sosfiltfilt,welch,resample_poly,iirnotch,filtfilt
from scipy.integrate import trapezoid
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.feature_selection import f_classif,mutual_info_classif
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score,confusion_matrix,ConfusionMatrixDisplay
from xgboost import XGBClassifier
from IPython.display import display
VERSIONS={p:version(p) for p in packages.values()}
print(platform.python_version(),VERSIONS)

print('DONE cell 2:',round(_stage_time.perf_counter()-_stage_start,2),'seconds',flush=True)

import pywt


In [ ]:
import time as _stage_time
print('START cell 4',flush=True)
_stage_start=_stage_time.perf_counter()
SEED=42; FS=500; CHANNELS=[0,3]; WINDOW=2500; STEP=1250
BANDS={'delta':(1,4),'theta':(4,8),'alpha':(8,12),'beta':(12,30),'gamma':(30,45)}
FILTER_ORDER=4; N_SELECTED=20; BALANCE=True; LABELS=np.array([-1,0,1])
SUBJECT_IDS=None  # None: all 15; [1,7] for a quick affected-subject check
SUSPECTED_TRIALS={(1,6),(7,21)}  # original one-based trial IDs; no automatic additions
SIGNAL_UNIT='uV'  # assumed, not independently verified
REMOVE_TRIAL_CONSTANT=True; ADDITIONAL_NOTCH=False
RUN_SINGLEM=True  # False for handcrafted-only execution
SINGLEM_ADDITIONAL_BANDPASS=True
SINGLEM_REPO=Path('/kaggle/working/SingLEM') if Path('/kaggle/working').exists() else Path.cwd()/'SingLEM'
SINGLEM_REF=None
ARCHIVE_PATH=None
DRIVE_ID='1Ma1fHzvhwyQm2zbtYXV8H-SACrmGH88L'
WORK=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()/'Q2_Wavelet_Comparison'
OUT=WORK/'results'; OUT.mkdir(parents=True,exist_ok=True)
if ARCHIVE_PATH is None and not Path('/kaggle/working').exists():
    local_archive=Path.cwd()/'LMB_05_HW1.zip'
    if local_archive.exists():ARCHIVE_PATH=str(local_archive)
config=dict(seed=SEED,fs=FS,channels_zero_based=CHANNELS,window=WINDOW,step=STEP,bands=BANDS,
    n_selected=N_SELECTED,balance=BALANCE,suspected_trials=sorted(SUSPECTED_TRIALS),
    signal_unit_assumed=SIGNAL_UNIT,units_independently_verified=False,
    remove_trial_constant=REMOVE_TRIAL_CONSTANT,additional_notch=ADDITIONAL_NOTCH,
    primary_evaluation='identical nonsuspected validation trials for both training policies',
    secondary_evaluation='suspected validation trials reported separately',versions=VERSIONS)
print('Results directory:',OUT,flush=True)

print('DONE cell 4:',round(_stage_time.perf_counter()-_stage_start,2),'seconds',flush=True)

BASE_OUT=OUT
config['pywavelets_module_version']=pywt.__version__
config['wavelet']=dict(wavelet='sym4',level=7,threshold_robust_sd=6,padding='reflect 1024 then multiple of 128',approximation='unchanged')


In [ ]:
import time as _stage_time
print('START cell 6',flush=True)
_stage_start=_stage_time.perf_counter()
if ARCHIVE_PATH is not None:
    archive=Path(ARCHIVE_PATH)
else:
    candidates=sorted(Path('/kaggle/input').rglob('LMB_05_HW1.zip')) if Path('/kaggle/input').exists() else []
    if len(candidates)>1: raise ValueError('Set ARCHIVE_PATH: multiple ZIPs found.')
    archive=candidates[0] if candidates else WORK/'LMB_05_HW1.zip'
if not archive.exists():
    if importlib.util.find_spec('gdown') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','gdown'])
    import gdown
    gdown.download(id=DRIVE_ID,output=str(archive),quiet=False)
assert zipfile.is_zipfile(archive), 'Expected ZIP, not an HTML access error.'
config['archive_sha256']=hashlib.sha256(archive.read_bytes()).hexdigest()
subjects={}
with zipfile.ZipFile(archive) as z:
    names=z.namelist()
    for name in [n for n in names if n.endswith('.emotion_preprocessed_epoch_InVideo_new.mat')]:
        folder=name.rsplit('/',1)[0]; sid=int(folder.rsplit('/',1)[-1])
        label_files=[n for n in names if n.startswith(folder+'/') and n.endswith('labels.mat')]
        assert len(label_files)==1,(sid,label_files)
        cell=loadmat(io.BytesIO(z.read(name)))['Preprocessed_trials']
        y=loadmat(io.BytesIO(z.read(label_files[0])))['matching_ratings'].ravel()
        assert cell.shape==(1,21) and y.shape==(21,) and np.isin(y,LABELS).all()
        trials=np.stack([np.asarray(t,dtype=np.float64) for t in cell.ravel()])
        assert trials.shape==(21,6,5000),sid
        assert np.isfinite(trials[:,CHANNELS,:]).all(),f'Selected ear contains nonfinite data: {sid}'
        assert sid not in subjects
        subjects[sid]={'trials':trials[:,CHANNELS,:],'y':y.astype(int), 'unused_nonfinite':int((~np.isfinite(trials)).sum())}
assert sorted(subjects)==[1,3,5,7,9,11,13,15,17,19,21,23,27,29,31]
subjects=dict(sorted(subjects.items()))
inventory=pd.DataFrame([dict(subject=sid,trials=21,unused_channel_nonfinite=d['unused_nonfinite'],**{f'label_{c}':int((d['y']==c).sum()) for c in LABELS}) for sid,d in subjects.items()])
display(inventory)
print('Archive SHA-256:',config['archive_sha256'])

assert SIGNAL_UNIT in ['V','mV','uV']
nb,na=iirnotch(50,30,fs=FS)
for sid,d in subjects.items():
    d['original_selected_ears']=d['trials'].copy()
    d['original_trial_numbers']=np.arange(1,22)
    if REMOVE_TRIAL_CONSTANT:d['trials']=d['trials']-d['trials'].mean(axis=-1,keepdims=True)
    if ADDITIONAL_NOTCH:d['trials']=filtfilt(nb,na,d['trials'],axis=-1)
    d['segments']=np.stack([trial[:,start:start+2500] for trial in d['trials'] for start in [0,1250,2500]])
    d['segment_y']=np.repeat(d['y'],3)
    d['trial_ids']=np.repeat(d['original_trial_numbers'],3)
    assert d['segments'].shape==(63,2,2500) and np.isfinite(d['segments']).all()
active_ids=sorted(subjects) if SUBJECT_IDS is None else list(SUBJECT_IDS)
assert active_ids and set(active_ids)<=set(subjects)
assert SUSPECTED_TRIALS <= {(sid,t) for sid in subjects for t in range(1,22)}
print('Original data retained:',len(subjects),'subjects,',sum(len(d['y']) for d in subjects.values()),'trials.',flush=True)

print('DONE cell 6:',round(_stage_time.perf_counter()-_stage_start,2),'seconds',flush=True)


In [ ]:
import time as _stage_time
print('START cell 8',flush=True)
_stage_start=_stage_time.perf_counter()
FROZEN_FOLD_IDS={1: [{'fold': 1, 'train_trials': [2, 3, 4, 5, 8, 11, 13, 14, 15, 16, 17, 18, 20, 21], 'validation_trials': [1, 6, 7, 9, 10, 12, 19]}, {'fold': 2, 'train_trials': [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 18, 19], 'validation_trials': [2, 14, 15, 16, 17, 20, 21]}, {'fold': 3, 'train_trials': [1, 2, 6, 7, 9, 10, 12, 14, 15, 16, 17, 19, 20, 21], 'validation_trials': [3, 4, 5, 8, 11, 13, 18]}], 3: [{'fold': 1, 'train_trials': [3, 6, 7, 8, 9, 10, 12, 14, 15, 16, 17, 18, 19, 20], 'validation_trials': [1, 2, 4, 5, 11, 13, 21]}, {'fold': 2, 'train_trials': [1, 2, 3, 4, 5, 7, 11, 13, 15, 16, 18, 19, 20, 21], 'validation_trials': [6, 8, 9, 10, 12, 14, 17]}, {'fold': 3, 'train_trials': [1, 2, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 17, 21], 'validation_trials': [3, 7, 15, 16, 18, 19, 20]}], 5: [{'fold': 1, 'train_trials': [2, 6, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20], 'validation_trials': [1, 3, 4, 5, 7, 12, 21]}, {'fold': 2, 'train_trials': [1, 3, 4, 5, 6, 7, 11, 12, 13, 15, 18, 19, 20, 21], 'validation_trials': [2, 8, 9, 10, 14, 16, 17]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 5, 7, 8, 9, 10, 12, 14, 16, 17, 21], 'validation_trials': [6, 11, 13, 15, 18, 19, 20]}], 7: [{'fold': 1, 'train_trials': [3, 6, 7, 8, 9, 10, 12, 13, 14, 15, 17, 18, 19, 20], 'validation_trials': [1, 2, 4, 5, 11, 16, 21]}, {'fold': 2, 'train_trials': [1, 2, 3, 4, 5, 7, 11, 14, 15, 16, 18, 19, 20, 21], 'validation_trials': [6, 8, 9, 10, 12, 13, 17]}, {'fold': 3, 'train_trials': [1, 2, 4, 5, 6, 8, 9, 10, 11, 12, 13, 16, 17, 21], 'validation_trials': [3, 7, 14, 15, 18, 19, 20]}], 9: [{'fold': 1, 'train_trials': [3, 6, 7, 8, 9, 12, 13, 14, 15, 16, 17, 18, 20, 21], 'validation_trials': [1, 2, 4, 5, 10, 11, 19]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 7, 8, 10, 11, 14, 15, 16, 18, 19, 20], 'validation_trials': [3, 6, 9, 12, 13, 17, 21]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 5, 6, 9, 10, 11, 12, 13, 17, 19, 21], 'validation_trials': [7, 8, 14, 15, 16, 18, 20]}], 11: [{'fold': 1, 'train_trials': [2, 3, 4, 6, 8, 9, 10, 12, 15, 17, 18, 19, 20, 21], 'validation_trials': [1, 5, 7, 11, 13, 14, 16]}, {'fold': 2, 'train_trials': [1, 3, 4, 5, 6, 7, 9, 10, 11, 13, 14, 15, 16, 18], 'validation_trials': [2, 8, 12, 17, 19, 20, 21]}, {'fold': 3, 'train_trials': [1, 2, 5, 7, 8, 11, 12, 13, 14, 16, 17, 19, 20, 21], 'validation_trials': [3, 4, 6, 9, 10, 15, 18]}], 13: [{'fold': 1, 'train_trials': [3, 4, 6, 7, 9, 10, 12, 13, 14, 17, 18, 19, 20, 21], 'validation_trials': [1, 2, 5, 8, 11, 15, 16]}, {'fold': 2, 'train_trials': [1, 2, 3, 4, 5, 7, 8, 11, 13, 14, 15, 16, 18, 19], 'validation_trials': [6, 9, 10, 12, 17, 20, 21]}, {'fold': 3, 'train_trials': [1, 2, 5, 6, 8, 9, 10, 11, 12, 15, 16, 17, 20, 21], 'validation_trials': [3, 4, 7, 13, 14, 18, 19]}], 15: [{'fold': 1, 'train_trials': [3, 6, 7, 9, 10, 12, 13, 14, 15, 16, 17, 18, 20, 21], 'validation_trials': [1, 2, 4, 5, 8, 11, 19]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 6, 7, 8, 11, 14, 15, 16, 18, 19, 20], 'validation_trials': [3, 9, 10, 12, 13, 17, 21]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 5, 8, 9, 10, 11, 12, 13, 17, 19, 21], 'validation_trials': [6, 7, 14, 15, 16, 18, 20]}], 17: [{'fold': 1, 'train_trials': [1, 3, 5, 6, 7, 8, 12, 13, 14, 16, 17, 19, 20, 21], 'validation_trials': [2, 4, 9, 10, 11, 15, 18]}, {'fold': 2, 'train_trials': [2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 14, 15, 18, 21], 'validation_trials': [1, 5, 13, 16, 17, 19, 20]}, {'fold': 3, 'train_trials': [1, 2, 4, 5, 9, 10, 11, 13, 15, 16, 17, 18, 19, 20], 'validation_trials': [3, 6, 7, 8, 12, 14, 21]}], 19: [{'fold': 1, 'train_trials': [3, 7, 8, 9, 10, 12, 14, 15, 16, 17, 18, 19, 20, 21], 'validation_trials': [1, 2, 4, 5, 6, 11, 13]}, {'fold': 2, 'train_trials': [1, 2, 3, 4, 5, 6, 7, 9, 11, 13, 14, 16, 18, 20], 'validation_trials': [8, 10, 12, 15, 17, 19, 21]}, {'fold': 3, 'train_trials': [1, 2, 4, 5, 6, 8, 10, 11, 12, 13, 15, 17, 19, 21], 'validation_trials': [3, 7, 9, 14, 16, 18, 20]}], 21: [{'fold': 1, 'train_trials': [3, 6, 7, 8, 9, 11, 12, 13, 14, 15, 17, 18, 20, 21], 'validation_trials': [1, 2, 4, 5, 10, 16, 19]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 7, 10, 12, 13, 14, 15, 16, 18, 19, 20], 'validation_trials': [3, 6, 8, 9, 11, 17, 21]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 16, 17, 19, 21], 'validation_trials': [7, 12, 13, 14, 15, 18, 20]}], 23: [{'fold': 1, 'train_trials': [1, 3, 6, 7, 9, 10, 12, 14, 15, 16, 17, 19, 20, 21], 'validation_trials': [2, 4, 5, 8, 11, 13, 18]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 8, 9, 11, 12, 13, 14, 15, 16, 18, 20], 'validation_trials': [3, 6, 7, 10, 17, 19, 21]}, {'fold': 3, 'train_trials': [2, 3, 4, 5, 6, 7, 8, 10, 11, 13, 17, 18, 19, 21], 'validation_trials': [1, 9, 12, 14, 15, 16, 20]}], 27: [{'fold': 1, 'train_trials': [3, 5, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21], 'validation_trials': [1, 2, 4, 6, 7, 8, 9]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 6, 7, 8, 9, 10, 16, 17, 18, 19, 21], 'validation_trials': [3, 11, 12, 13, 14, 15, 20]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 6, 7, 8, 9, 11, 12, 13, 14, 15, 20], 'validation_trials': [5, 10, 16, 17, 18, 19, 21]}], 29: [{'fold': 1, 'train_trials': [3, 6, 7, 9, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21], 'validation_trials': [1, 2, 4, 5, 8, 10, 11]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 6, 7, 8, 10, 11, 14, 16, 18, 19, 21], 'validation_trials': [3, 9, 12, 13, 15, 17, 20]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 5, 8, 9, 10, 11, 12, 13, 15, 17, 20], 'validation_trials': [6, 7, 14, 16, 18, 19, 21]}], 31: [{'fold': 1, 'train_trials': [1, 2, 5, 6, 7, 9, 10, 12, 14, 15, 16, 17, 20, 21], 'validation_trials': [3, 4, 8, 11, 13, 18, 19]}, {'fold': 2, 'train_trials': [1, 2, 3, 4, 5, 6, 8, 10, 11, 12, 13, 18, 19, 21], 'validation_trials': [7, 9, 14, 15, 16, 17, 20]}, {'fold': 3, 'train_trials': [3, 4, 7, 8, 9, 11, 13, 14, 15, 16, 17, 18, 19, 20], 'validation_trials': [1, 2, 5, 6, 10, 12, 21]}]}
ORIGINAL_FOLD_FILE_SHA256='d657d62bc72561bf7c6abaffcd9df319eab91f61ce129d665ba101ed4adfebc4'

config['original_fold_file_sha256']=ORIGINAL_FOLD_FILE_SHA256
folds={};split_rows=[]
for sid in active_ids:
    d=subjects[sid];seen=[];folds[sid]=[]
    for item in FROZEN_FOLD_IDS[sid]:
        train_ids=np.asarray(item['train_trials'],int);val_ids=np.asarray(item['validation_trials'],int)
        assert not set(train_ids)&set(val_ids)
        assert set(train_ids)|set(val_ids)==set(range(1,22))
        seen.extend(val_ids.tolist())
        suspected_ids={t for s,t in SUSPECTED_TRIALS if s==sid}
        train=np.flatnonzero(np.isin(d['trial_ids'],train_ids))
        train_without=np.flatnonzero(np.isin(d['trial_ids'],[t for t in train_ids if t not in suspected_ids]))
        val_all=np.flatnonzero(np.isin(d['trial_ids'],val_ids))
        val_common=np.flatnonzero(np.isin(d['trial_ids'],[t for t in val_ids if t not in suspected_ids]))
        val_suspect=np.flatnonzero(np.isin(d['trial_ids'],[t for t in val_ids if t in suspected_ids]))
        assert len(train)==42 and len(val_all)==21 and len(val_common)>0
        assert set(val_all)==set(val_common)|set(val_suspect)
        assert not set(val_common)&set(val_suspect)
        assert not set(d['trial_ids'][train_without])&suspected_ids
        fold=dict(fold=item['fold'],train_include=train,train_exclude=train_without,
                  val_all=val_all,val_common=val_common,val_suspect=val_suspect)
        folds[sid].append(fold)
        split_rows.append(dict(subject=sid,fold=item['fold'],
            train_include_trials=json.dumps(train_ids.tolist()),
            train_exclude_trials=json.dumps(np.unique(d['trial_ids'][train_without]).tolist()),
            common_validation_trials=json.dumps(np.unique(d['trial_ids'][val_common]).tolist()),
            suspected_validation_trials=json.dumps(np.unique(d['trial_ids'][val_suspect]).tolist()),
            include_training_windows=len(train),exclude_training_windows=len(train_without),
            common_validation_windows=len(val_common),suspected_validation_windows=len(val_suspect),
            training_data_changed=not np.array_equal(train,train_without),
            **{f'include_train_label_{c}':int((d['segment_y'][train]==c).sum()) for c in LABELS},
            **{f'exclude_train_label_{c}':int((d['segment_y'][train_without]==c).sum()) for c in LABELS},
            **{f'common_val_label_{c}':int((d['segment_y'][val_common]==c).sum()) for c in LABELS}))
    assert sorted(seen)==list(range(1,22))
split_audit=pd.DataFrame(split_rows);split_audit.to_csv(OUT/'fixed_fold_comparison_audit.csv',index=False)
display(split_audit[split_audit.subject.isin([s for s,t in SUSPECTED_TRIALS])])
print('Folds frozen. No split regeneration.',flush=True)

print('DONE cell 8:',round(_stage_time.perf_counter()-_stage_start,2),'seconds',flush=True)


In [ ]:
# Verify saved assignments against actual StratifiedKFold, not merely disjointness.
cv_rows=[]
for sid in active_ids:
    d=subjects[sid]
    generated=list(StratifiedKFold(3,shuffle=True,random_state=SEED).split(np.zeros(21),d['y']))
    for item,(tr,va) in zip(FROZEN_FOLD_IDS[sid],generated):
        assert np.array_equal(np.sort(tr+1),np.sort(item['train_trials']))
        assert np.array_equal(np.sort(va+1),np.sort(item['validation_trials']))
        for label in LABELS:
            cv_rows.append(dict(subject=sid,fold=item['fold'],label=int(label),train_trials=int((d['y'][tr]==label).sum()),validation_trials=int((d['y'][va]==label).sum()),total_trials=int((d['y']==label).sum())))
cv_audit=pd.DataFrame(cv_rows);cv_audit.to_csv(BASE_OUT/'stratification_audit.csv',index=False)
assert set(cv_audit.loc[cv_audit.total_trials<3,'subject'])=={17,31}
print('PASS: every frozen fold exactly matches seed-42 trial-level StratifiedKFold; rare-class limitations: 17,31')

def wavelet_filter(x):
    x=np.asarray(x,dtype=float); n=len(x); pad=1024
    padded=np.pad(x,(pad,pad+(-(n+2*pad))%128),mode='reflect')
    coefficients=pywt.swt(padded,'sym4',level=7,trim_approx=True,norm=True)
    rebuilt=pywt.iswt(coefficients,'sym4',norm=True)[pad:pad+n]
    assert np.allclose(rebuilt,x,rtol=1e-9,atol=1e-7)
    cleaned=[coefficients[0]]
    for detail in coefficients[1:]:
        core=detail[pad:pad+n]; center=np.median(core)
        sigma=np.median(abs(core-center))/0.6744897501960817
        cleaned.append(np.clip(detail,center-6*sigma,center+6*sigma) if sigma>np.finfo(float).eps else detail.copy())
    result=pywt.iswt(cleaned,'sym4',norm=True)[pad:pad+n]
    return result-result.mean()

baseline_trials={sid:d['trials'].copy() for sid,d in subjects.items()}
wavelet_trials={sid:np.stack([np.stack([wavelet_filter(x) for x in trial]) for trial in d['trials']]) for sid,d in subjects.items()}
quality=[]
for sid in active_ids:
    for ti,(before,after) in enumerate(zip(baseline_trials[sid],wavelet_trials[sid]),1):
        for ear,(x,z) in enumerate(zip(before,after)):
            row=dict(subject=sid,trial=ti,channel=[1,4][ear],label=int(subjects[sid]['y'][ti-1]),severe_suspect=(sid,ti) in SUSPECTED_TRIALS,
                peak_before=float(abs(x).max()),peak_after=float(abs(z).max()),rms_before=float(np.sqrt(np.mean(x*x))),rms_after=float(np.sqrt(np.mean(z*z))),
                percent_over_100_before=float(100*np.mean(abs(x)>100)),percent_over_100_after=float(100*np.mean(abs(z)>100)),
                removed_energy_fraction=float(np.sum((x-z)**2)/max(np.sum(x*x),1e-30)),correlation=float(np.corrcoef(x,z)[0,1]))
            f,p=welch(x,fs=FS,nperseg=1000);_,q=welch(z,fs=FS,nperseg=1000)
            for band,(lo,hi) in BANDS.items():
                mask=(f>=lo)&(f<=hi); row[band+'_power_ratio']=float(trapezoid(q[mask],f[mask])/max(trapezoid(p[mask],f[mask]),1e-30))
            quality.append(row)
quality=pd.DataFrame(quality);quality.to_csv(BASE_OUT/'wavelet_quality_all_trials.csv',index=False)
flagged={(1,6),(1,7),(3,3),(3,14),(3,21),(5,6),(7,21),(13,8),(13,10),(13,12),(13,21),(15,12),(17,19),(19,1),(21,19),(21,20),(29,9),(31,5)}
quality[[ (s,t) in flagged for s,t in zip(quality.subject,quality.trial)]].to_csv(BASE_OUT/'suspected_trials_before_after.csv',index=False)
for sid,ti in sorted(flagged):
    fig,axs=plt.subplots(2,2,figsize=(13,6))
    for ear in range(2):
        x=baseline_trials[sid][ti-1,ear];z=wavelet_trials[sid][ti-1,ear];t=np.arange(len(x))/FS
        axs[ear,0].plot(t,x,label='baseline',alpha=.65);axs[ear,0].plot(t,z,label='wavelet',alpha=.7)
        axs[ear,1].plot(t,x-z,label='removed component');axs[ear,0].set_ylabel(f'Channel {[1,4][ear]} native units')
        for ax in axs[ear]:ax.legend();ax.set_xlabel('seconds')
    fig.suptitle(f'Subject {sid}, trial {ti}: full amplitude axes');fig.tight_layout();fig.savefig(BASE_OUT/f'waveform_{sid}_{ti}.png',dpi=130);plt.close(fig)


In [ ]:
import time as _stage_time
print('START cell 10',flush=True)
_stage_start=_stage_time.perf_counter()
STARTS=[0,1250,2500]
STAT_NAMES=['mean','std','diff1','norm_diff1','diff2','norm_diff2','relative_power','DE']
FEATURE_NAMES=[f'{ear}_{band}_{stat}' for ear in ['right','left'] for band in BANDS for stat in STAT_NAMES]
SOS={b:butter(FILTER_ORDER,limits,btype='bandpass',fs=FS,output='sos') for b,limits in BANDS.items()}
def integrate_psd(f,p,lo,hi):
    inside=(f>lo)&(f<hi)
    grid=np.r_[lo,f[inside],hi]
    return trapezoid(np.r_[np.interp(lo,f,p),p[inside],np.interp(hi,f,p)],grid)
def handcrafted_features(trials):
    rows=[]
    for trial in trials:
        filtered={b:sosfiltfilt(sos,trial,axis=-1) for b,sos in SOS.items()}
        for start in STARTS:
            row=[]
            for ear in range(2):
                raw=trial[ear,start:start+WINDOW]
                f,p=welch(raw,fs=FS,window='hann',nperseg=1000,noverlap=500,detrend='constant',scaling='density')
                den=integrate_psd(f,p,1,100)
                for band,(lo,hi) in BANDS.items():
                    x=filtered[band][ear,start:start+WINDOW]
                    mu,sd=x.mean(),x.std(ddof=0)
                    z=(x-mu)/sd if sd>0 else np.zeros_like(x)
                    rp=integrate_psd(f,p,lo,hi)/den if den>0 else 0.0
                    row.extend([mu,sd,np.abs(np.diff(x)).mean(),np.abs(np.diff(z)).mean(),
                                np.abs(x[2:]-x[:-2]).mean(),np.abs(z[2:]-z[:-2]).mean(),rp,
                                .5*np.log(2*np.pi*np.e*max(sd**2,1e-30))])
            rows.append(row)
    X=np.asarray(rows)
    assert X.shape==(len(trials)*3,80) and np.isfinite(X).all()
    return X
active_ids=sorted(subjects) if SUBJECT_IDS is None else list(SUBJECT_IDS)
assert active_ids and set(active_ids)<=set(subjects)


In [ ]:
import time as _stage_time
print('START cell 12',flush=True)
_stage_start=_stage_time.perf_counter()
def select_features(X,y,method,k):
    scores=np.zeros(X.shape[1]); variable=np.flatnonzero(np.var(X,axis=0)>0)
    if method=='MI': scores[variable]=mutual_info_classif(X[:,variable],y,discrete_features=False,random_state=SEED)
    elif method=='F-test':
        with warnings.catch_warnings():
            warnings.simplefilter('ignore'); scores[variable]=f_classif(X[:,variable],y)[0]
    else: raise ValueError(method)
    scores=np.nan_to_num(scores,nan=0.0,posinf=np.finfo(float).max,neginf=0.0)
    return np.argsort(-scores,kind='stable')[:min(k,X.shape[1])],scores

def balanced_rows(y,seed):
    rng=np.random.default_rng(seed); classes,counts=np.unique(y,return_counts=True); rows=[]
    for c,n in zip(classes,counts):
        idx=np.flatnonzero(y==c); rows.extend(idx.tolist())
        if n<counts.max(): rows.extend(rng.choice(idx,size=counts.max()-n,replace=True).tolist())
    return rng.permutation(rows)

def make_model(name,n_classes):
    if name=='SVM': return SVC(C=1.0,kernel='rbf',gamma='scale')
    if name=='LDA': return LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto')
    if name=='Decision Tree': return DecisionTreeClassifier(max_depth=3,min_samples_leaf=2,random_state=SEED)
    if name=='XGBoost':
        kw=dict(n_estimators=100,max_depth=2,learning_rate=.05,reg_lambda=1.0,subsample=1.0,
                colsample_bytree=1.0,tree_method='hist',n_jobs=1,random_state=SEED,verbosity=0)
        if n_classes>2: kw.update(objective='multi:softprob',num_class=n_classes,eval_metric='mlogloss')
        else: kw.update(objective='binary:logistic',eval_metric='logloss')
        return XGBClassifier(**kw)
    raise ValueError(name)
MODEL_NAMES=['SVM','LDA','Decision Tree','XGBoost']

print('DONE cell 12:',round(_stage_time.perf_counter()-_stage_start,2),'seconds',flush=True)


In [ ]:
import time as _stage_time
print('START cell 14',flush=True)
_stage_start=_stage_time.perf_counter()
def paired_evaluate(feature_map,feature_names,representation):
    score_rows=[];prediction_rows=[];suspect_rows=[];original_rows=[];feature_rows=[]
    fit_count=0;reuse_count=0
    for sid in active_ids:
        d=subjects[sid];X=np.asarray(feature_map[sid],float);y=d['segment_y']
        assert X.shape==(63,len(feature_names)) and np.isfinite(X).all()
        for f in folds[sid]:
            fold=f['fold'];common=f['val_common'];suspect=f['val_suspect'];val_all=f['val_all']
            for selector in ['MI','F-test']:
                for model_name in MODEL_NAMES:
                    include_fit=None;include_common_pred=None
                    for policy in ['include','exclude']:
                        tr=f['train_'+policy]
                        reused=policy=='exclude' and np.array_equal(tr,f['train_include'])
                        if reused:
                            scaler,encoder,chosen,fscores,model=include_fit;reuse_count+=1
                        else:
                            scaler=StandardScaler().fit(X[tr])
                            Xt=scaler.transform(X[tr])
                            encoder=LabelEncoder().fit(y[tr]);yt=encoder.transform(y[tr])
                            if len(encoder.classes_)<2:raise ValueError(f'{sid}/{fold}/{policy}: fewer than two training classes')
                            chosen,fscores=select_features(Xt,yt,selector,N_SELECTED)
                            rows=balanced_rows(yt,SEED+sid*10+fold) if BALANCE else np.arange(len(tr))
                            model=make_model(model_name,len(encoder.classes_))
                            model.fit(Xt[rows][:,chosen],yt[rows]);fit_count+=1
                        if policy=='include':include_fit=(scaler,encoder,chosen,fscores,model)
                        pred=encoder.inverse_transform(model.predict(scaler.transform(X[common])[:,chosen]).astype(int))
                        score=f1_score(y[common],pred,labels=LABELS,average='macro',zero_division=0)
                        if policy=='include':include_common_pred=pred.copy()
                        if reused:assert np.array_equal(pred,include_common_pred)
                        score_rows.append(dict(representation=representation,subject=sid,fold=fold,selector=selector,model=model_name,
                            policy=policy,macro_f1=score,common_validation_windows=len(common),training_windows=len(tr),fit_reused=reused))
                        for index,prediction in zip(common,pred):
                            prediction_rows.append(dict(representation=representation,subject=sid,fold=fold,selector=selector,model=model_name,
                                policy=policy,segment=int(index),trial=int(d['trial_ids'][index]),truth=int(y[index]),prediction=int(prediction)))
                        for rank,j in enumerate(chosen,1):
                            feature_rows.append(dict(representation=representation,subject=sid,fold=fold,selector=selector,model=model_name,
                                policy=policy,rank=rank,feature=feature_names[j],score=float(fscores[j]),fit_reused=reused))
                        if len(suspect):
                            sp=encoder.inverse_transform(model.predict(scaler.transform(X[suspect])[:,chosen]).astype(int))
                            for index,prediction in zip(suspect,sp):
                                suspect_rows.append(dict(representation=representation,subject=sid,fold=fold,selector=selector,model=model_name,
                                    policy=policy,segment=int(index),trial=int(d['trial_ids'][index]),truth=int(y[index]),prediction=int(prediction)))
                        if policy=='include':
                            original_pred=encoder.inverse_transform(model.predict(scaler.transform(X[val_all])[:,chosen]).astype(int))
                            original_rows.append(dict(representation=representation,subject=sid,fold=fold,selector=selector,model=model_name,
                                original_full_validation_macro_f1=f1_score(y[val_all],original_pred,labels=LABELS,average='macro',zero_division=0)))
        print(representation,'subject',sid,'complete; cumulative fits',fit_count,'reused pipelines',reuse_count,flush=True)
    scores=pd.DataFrame(score_rows);predictions=pd.DataFrame(prediction_rows)
    suspected=pd.DataFrame(suspect_rows,columns=['representation','subject','fold','selector','model','policy','segment','trial','truth','prediction'])
    originals=pd.DataFrame(original_rows);features=pd.DataFrame(feature_rows)
    keys=['representation','subject','fold','selector','model','segment','trial','truth']
    a=predictions.query("policy=='include'");b=predictions.query("policy=='exclude'")
    paired=a.merge(b,on=keys,validate='one_to_one',suffixes=('_include','_exclude'))
    assert len(paired)==len(a)==len(b),'Primary evaluation samples differ between policies.'
    assert not set(zip(predictions.subject,predictions.trial))&SUSPECTED_TRIALS
    assert not predictions.duplicated(['subject','selector','model','policy','segment']).any()
    for sid in active_ids:
        expected=3*(21-sum(s==sid for s,t in SUSPECTED_TRIALS))
        counts=predictions[predictions.subject==sid].groupby(['selector','model','policy']).size()
        assert counts.eq(expected).all()
    for name,table in [('paired_fold_scores',scores),('common_validation_predictions',predictions),
        ('suspected_validation_predictions',suspected),('original_full_validation_fold_scores',originals),('selected_features',features)]:
        table.to_csv(OUT/f'{representation}_{name}.csv',index=False)
    config[representation+'_actual_model_fits']=fit_count;config[representation+'_reused_model_fits']=reuse_count
    return scores,predictions,suspected,originals



In [ ]:
import time as _stage_time
print('START cell 16',flush=True)
_stage_start=_stage_time.perf_counter()
def summarize_paired(scores,suspected,originals,tag):
    subject=scores.groupby(['representation','subject','selector','model','policy'],as_index=False).macro_f1.mean()
    wide=subject.pivot(index=['representation','subject','selector','model'],columns='policy',values='macro_f1').reset_index()
    wide.columns.name=None
    wide=wide.rename(columns={'include':'include_common_macro_f1','exclude':'exclude_common_macro_f1'})
    wide['delta_exclude_minus_include']=wide.exclude_common_macro_f1-wide.include_common_macro_f1
    wide['delta_percentage_points']=100*wide.delta_exclude_minus_include
    affected={s for s,t in SUSPECTED_TRIALS}
    wide['subject_has_suspected_trial']=wide.subject.isin(affected)
    assert np.allclose(wide.loc[~wide.subject_has_suspected_trial,'delta_exclude_minus_include'],0,atol=0,rtol=0)
    cohort=wide.groupby(['representation','selector','model']).agg(
        mean_include_common_macro_f1=('include_common_macro_f1','mean'),
        mean_exclude_common_macro_f1=('exclude_common_macro_f1','mean'),
        mean_paired_delta=('delta_exclude_minus_include','mean'),
        sd_paired_delta_between_subjects=('delta_exclude_minus_include','std'),
        n_subjects=('subject','nunique')).reset_index()
    affected_means=wide[wide.subject_has_suspected_trial].groupby(['representation','selector','model']).agg(
        mean_delta_affected_subjects=('delta_exclude_minus_include','mean'),n_affected_subjects=('subject','nunique')).reset_index()
    cohort=cohort.merge(affected_means,on=['representation','selector','model'],how='left')
    focus=wide[(wide.selector=='F-test')&(wide.model=='SVM')].copy()
    original=originals.groupby(['representation','subject','selector','model'],as_index=False).original_full_validation_macro_f1.mean()
    wide=wide.merge(original,on=['representation','subject','selector','model'],validate='one_to_one')
    focus=focus.merge(original,on=['representation','subject','selector','model'],validate='one_to_one')
    if len(suspected):
        suspected=suspected.copy();suspected['correct']=suspected.truth==suspected.prediction
        secondary=suspected.groupby(['representation','subject','trial','selector','model','policy'],as_index=False).agg(
            n_windows=('correct','size'),correct_windows=('correct','sum'),accuracy=('correct','mean'))
    else:secondary=pd.DataFrame(columns=['representation','subject','trial','selector','model','policy','n_windows','correct_windows','accuracy'])
    for name,table in [('paired_subject_results',wide),('paired_cohort_summary',cohort),('focus_Ftest_SVM',focus),
                       ('suspected_trial_diagnostics',secondary)]:table.to_csv(OUT/f'{tag}_{name}.csv',index=False)
    print(tag,'F-test + SVM: identical common validation trials in both columns')
    display(focus)
    print(tag,'affected subjects only:');display(focus[focus.subject_has_suspected_trial])
    fig,ax=plt.subplots(figsize=(11,4))
    ax.bar(focus.subject.astype(str),focus.delta_percentage_points,
           color=np.where(focus.delta_percentage_points>=0,'#00897b','#c62828'))
    ax.axhline(0,color='black',lw=.8)
    ax.set(xlabel='Subject',ylabel='EXCLUDE minus INCLUDE (percentage points)',title=f'{tag}: fixed-fold F-test + SVM on common validation trials')
    fig.tight_layout();fig.savefig(OUT/f'{tag}_paired_Ftest_SVM_delta.png',dpi=160);plt.show();plt.close(fig)
    return wide,cohort,focus,secondary


In [ ]:
all_results=[]
for condition,trial_map in [('baseline',baseline_trials),('wavelet',wavelet_trials)]:
    OUT=BASE_OUT/condition;OUT.mkdir(exist_ok=True)
    for sid in active_ids:
        subjects[sid]['trials']=trial_map[sid]
        subjects[sid]['X']=handcrafted_features(trial_map[sid])
    hand_scores,hand_predictions,hand_suspect_predictions,hand_original_scores=paired_evaluate({sid:subjects[sid]['X'] for sid in active_ids},FEATURE_NAMES,'handcrafted')
    hand_subject_results,hand_cohort_summary,hand_focus,hand_secondary=summarize_paired(hand_scores,hand_suspect_predictions,hand_original_scores,'handcrafted')
    hand_subject_results['condition']=condition;all_results.append(hand_subject_results.copy())
    import time as _stage_time
    print('START cell 18',flush=True)
    _stage_start=_stage_time.perf_counter()
    if RUN_SINGLEM:
        import torch,importlib.util
        if not SINGLEM_REPO.exists():
            subprocess.check_call(['git','clone','https://github.com/ttlabtuat/SingLEM.git',str(SINGLEM_REPO)])
        if SINGLEM_REF:subprocess.check_call(['git','-C',str(SINGLEM_REPO),'checkout',SINGLEM_REF])
        config['singlem_commit']=subprocess.check_output(['git','-C',str(SINGLEM_REPO),'rev-parse','HEAD'],text=True).strip()
        checkpoint=SINGLEM_REPO/'SingLEM'/'checkpoints'/'singlem_downstream_excluded.pt'
        assert checkpoint.exists() and checkpoint.stat().st_size>10000
        spec=importlib.util.spec_from_file_location('fixed_fold_singlem',SINGLEM_REPO/'SingLEM'/'model.py')
        module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
        payload=torch.load(checkpoint,map_location='cpu',weights_only=True)
        assert {'model_config','encoder_state_dict'}<=payload.keys()
        cfg=module.Config()
        for key,value in payload['model_config'].items():setattr(cfg,key,value)
        cfg.mask_prob=0.0
        device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        encoder_net=module.EEGEncoder(cfg).to(device);encoder_net.load_state_dict(payload['encoder_state_dict'],strict=True);encoder_net.eval()
        for parameter in encoder_net.parameters():parameter.requires_grad_(False)
        config['singlem_checkpoint_sha256']=hashlib.sha256(checkpoint.read_bytes()).hexdigest()
        config['singlem_preparation']=dict(additional_bandpass=SINGLEM_ADDITIONAL_BANDPASS,unit_assumed=SIGNAL_UNIT,
            input_scale_from_volts=1e4,tokens='five nonoverlapping 1-s tokens per window',pooling='token mean per ear, concatenate ears')
        unit_scale={'V':1.,'mV':1e-3,'uV':1e-6}[SIGNAL_UNIT]
        broad=butter(4,[.5,50],fs=FS,btype='bandpass',output='sos')
        embedding_map={}
        with torch.inference_mode():
            for sid in active_ids:
                volt=subjects[sid]['trials']*unit_scale
                prepared=sosfiltfilt(broad,volt,axis=-1) if SINGLEM_ADDITIONAL_BANDPASS else volt
                if (abs(prepared)>100e-6).any():warnings.warn(f'Subject {sid}: above the 100-uV diagnostic reference; retained for paired comparison.')
                sampled=resample_poly(prepared,32,125,axis=-1);assert sampled.shape==(21,2,1280)
                windows=np.stack([trial[:,start:start+640] for trial in sampled for start in [0,320,640]])
                ear_vectors=[]
                for ear in [0,1]:
                    tokens=torch.tensor(windows[:,ear,:].reshape(63,5,128)*1e4,dtype=torch.float32,device=device)
                    parts=[]
                    for batch in tokens.split(16):
                        reps=encoder_net(batch)[0];assert tuple(reps.shape[1:])==(5,16)
                        parts.append(reps.mean(dim=1).cpu().numpy())
                    ear_vectors.append(np.concatenate(parts))
                embedding_map[sid]=np.concatenate(ear_vectors,axis=1)
                assert embedding_map[sid].shape==(63,32) and np.isfinite(embedding_map[sid]).all()
                print('SingLEM embeddings:',sid,embedding_map[sid].shape,flush=True)
        embedding_names=[f'{ear}_embedding_{i}' for ear in ['right','left'] for i in range(16)]
        emb_scores,emb_predictions,emb_suspected,emb_originals=paired_evaluate(embedding_map,embedding_names,'singlem')
        emb_subject_results,emb_cohort_summary,emb_focus,emb_secondary=summarize_paired(emb_scores,emb_suspected,emb_originals,'singlem')
    else:print('RUN_SINGLEM=False: only handcrafted comparison executed. No SingLEM results are invented.',flush=True)
    
    print('DONE cell 18:',round(_stage_time.perf_counter()-_stage_start,2),'seconds',flush=True)


In [ ]:
OUT=BASE_OUT
combined=pd.concat(all_results,ignore_index=True);combined.to_csv(OUT/'all_subject_results.csv',index=False)
keys=['subject','selector','model']
paired=combined.query("condition=='baseline'").merge(combined.query("condition=='wavelet'"),on=keys,suffixes=('_baseline','_wavelet'),validate='one_to_one')
for policy in ['include','exclude']:
    metric=policy+'_common_macro_f1'
    paired[policy+'_delta_pp']=100*(paired[metric+'_wavelet']-paired[metric+'_baseline'])
paired.to_csv(OUT/'wavelet_paired_subject_comparison.csv',index=False)
summary=paired.groupby(['selector','model'])[[p+'_common_macro_f1_'+c for p in ['include','exclude'] for c in ['baseline','wavelet']]+['include_delta_pp','exclude_delta_pp']].mean().reset_index()
summary.to_csv(OUT/'wavelet_cohort_comparison.csv',index=False);display(summary)
if RUN_SINGLEM:
    sb=pd.read_csv(BASE_OUT/'baseline/singlem_paired_subject_results.csv')
    sw=pd.read_csv(BASE_OUT/'wavelet/singlem_paired_subject_results.csv')
    sp=sb.merge(sw,on=keys,suffixes=('_baseline','_wavelet'),validate='one_to_one')
    for policy in ['include','exclude']:
        metric=policy+'_common_macro_f1'
        sp[policy+'_delta_pp']=100*(sp[metric+'_wavelet']-sp[metric+'_baseline'])
    sp.to_csv(OUT/'singlem_wavelet_paired_subject_comparison.csv',index=False)
    ss=sp.groupby(['selector','model'])[[p+'_common_macro_f1_'+c for p in ['include','exclude'] for c in ['baseline','wavelet']]+['include_delta_pp','exclude_delta_pp']].mean().reset_index()
    ss.to_csv(OUT/'singlem_wavelet_cohort_comparison.csv',index=False);display(ss)
config.update(run_singlem=RUN_SINGLEM,active_subjects=active_ids)
(OUT/'configuration.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
with zipfile.ZipFile(WORK/'Wavelet_Comparison_Results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file():z.write(p,str(p.relative_to(OUT)))
